In [1]:
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd

from pytrader import li_currencies


# FTSE for GBP Pairs
def fn_ftse():
    # Define the ticker symbol for the FTSE 100
    print('Getting FTSE 100')
    ticker_symbol = "DX-Y.NYB"

    # Get data for the ticker
    df_ftse = yf.download(ticker_symbol,
                            start=(datetime.now() - relativedelta(days=60)).strftime('%Y-%m-%d'),
                            end=datetime.now().strftime('%Y-%m-%d'),
                            auto_adjust=True)

    df_ftse.columns = df_ftse.columns.droplevel(1)
    df_ftse.reset_index(drop=False,inplace=True)
    df_ftse = df_ftse.round(2)

    return df_ftse
# fn_ftse()

In [3]:
# Calculate DXY H1 H4
from importlib import reload
import oanda_pricing
from time import sleep
reload(oanda_pricing)

oanda_acct = ''
oanda_token = ''
oanda_url = 'api-fxpractice.oanda.com'

oanda = oanda_pricing.OandaPricing(oanda_acct=oanda_acct,
                                   oanda_token=oanda_token,
                                   oanda_url=oanda_url)

def fn_dxy(oanda):
    df = pd.DataFrame()
    li_dxy_currencies = ['EUR_USD', 'USD_JPY', 'GBP_USD', 'USD_CAD', 'USD_SEK', 'USD_CHF']

    for currency in li_dxy_currencies:
        print('DXY Currency: ' + currency)
        df_temp = oanda.fn_oanda_dxy(currency)
        df_temp['currency'] = currency
        df = pd.concat([df, df_temp], ignore_index=True)

    li_ohlc = ['open', 'high', 'low', 'close']

    df_pvt = df.pivot(index=['datetime', 'interval'],
                      columns='currency',
                      values=li_ohlc)

    for ohlc in li_ohlc:
        df_pvt[('dxy',ohlc)] = 50.14348112 * \
            (df_pvt[(ohlc,'EUR_USD')] ** -0.576) * \
            (df_pvt[(ohlc,'USD_JPY')] ** 0.136) * \
            (df_pvt[(ohlc,'GBP_USD')] ** -0.119) * \
            (df_pvt[(ohlc,'USD_CAD')] ** 0.091) * \
            (df_pvt[(ohlc,'USD_SEK')] ** 0.042) * \
            (df_pvt[(ohlc,'USD_CHF')] ** 0.036)

    df_pvt.drop(li_ohlc, axis=1, inplace=True)
    df_pvt.reset_index(inplace=True)

    df_pvt.columns = df_pvt.columns.get_level_values(1)
    df_pvt.columns.name = 'dxy'
    df_pvt.columns = ['datetime', 'interval', 'open', 'high', 'low', 'close']
    df_pvt.sort_values(['interval', 'datetime'], inplace=True)

    return df_pvt

fn_dxy(oanda)

DXY Currency: EUR_USD
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200
DXY Currency: USD_JPY
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200
DXY Currency: GBP_USD
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200
DXY Currency: USD_CAD
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200
DXY Currency: USD_SEK
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200
DXY Currency: USD_CHF
Historical Prices Response (Interval: M15): 200
Historical Prices Response (Interval: H1): 200
Historical Prices Response (Interval: H4): 200


,datetime,interval,open,high,low,close
48,2025-12-04 13:00:00,H1,98.955165,98.941914,98.965270,98.972539
50,2025-12-04 14:00:00,H1,98.973720,98.978097,98.992383,98.990655
51,2025-12-04 15:00:00,H1,98.991077,99.012833,99.057069,99.056845
52,2025-12-04 16:00:00,H1,99.056975,99.054717,99.070290,99.056595
53,2025-12-04 17:00:00,H1,99.043998,99.044818,99.047762,99.051469
55,2025-12-04 18:00:00,H1,99.051681,99.066988,99.072564,99.073023
56,2025-12-04 19:00:00,H1,99.073003,99.040517,99.048772,99.012365
57,2025-12-04 20:00:00,H1,99.012667,99.002357,99.004217,98.990694
58,2025-12-04 21:00:00,H1,98.989842,98.995700,99.002324,99.004198
60,2025-12-04 22:00:00,H1,99.003197,99.020377,99.034633,99.033257
